<a href="https://colab.research.google.com/github/Gunjannn07/FlyRank1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gunjannn07/FlyRank1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


I will prioritize pages that have an opportunity for improved CTR.

The rule uses two signals:

1. **Average search position and CTR:** pages with lower CTR than expected for their position may have an opportunity for CTR improvement.
2. **Search volume:** higher-volume pages have more potential impact if their CTR improves.

I will first check both signals using bucket tables before using them in the score.

### Rule logic

A page receives a higher score when:

- its CTR is below the typical CTR for its search-position bucket, and
- it has higher search volume.

##### Reason code: `CTR_OPPORTUNITY`

##### Action:`REVIEW_CTR`

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [4]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


**Signal check and rule setup**

In [6]:
import pandas as pd
import numpy as np

#CTR vs search position

position_bins = [0, 3, 10, 20, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21+"]

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

position_check = (
    df.groupby("position_bucket", observed=True)
      .agg(
          mean_ctr=("ctr", "mean"),
          n=("ctr", "count")
      )
      .reset_index()
)

print("SIGNAL 1: CTR vs SEARCH POSITION")
print(position_check.to_string(index=False))

print("\nPosition signal verdict: CONFIRMED")

# Search volume

volume_bins = [-np.inf, 0, 100, 1000, np.inf]
volume_labels = ["0", "1-100", "101-1000", "1000+"]

df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=volume_bins,
    labels=volume_labels,
    include_lowest=True
)

volume_check = (
    df.groupby("volume_bucket", observed=True)
      .agg(
          mean_ctr=("ctr", "mean"),
          mean_position=("avg_position", "mean"),
          n=("content_id", "count")
      )
      .reset_index()
)

print("\nSIGNAL 2: SEARCH VOLUME")
print(volume_check.to_string(index=False))

print("\nVolume signal verdict: CONFIRMED")

SIGNAL 1: CTR vs SEARCH POSITION
position_bucket  mean_ctr     n
            1-3  1.472869  2346
           4-10  0.651045 11842
          11-20  0.323443  7273
            21+  0.211333  8539

Position signal verdict: CONFIRMED

SIGNAL 2: SEARCH VOLUME
volume_bucket  mean_ctr  mean_position     n
            0  0.437774      16.047920 11081
        1-100  0.255963      17.157506 13402
     101-1000  0.182081      20.339172  2489
        1000+  0.182089      23.678750   560

Volume signal verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
expected_ctr = (
    df.groupby("position_bucket", observed=True)["ctr"]
      .mean()
)

df["expected_ctr"] = (
    df["position_bucket"]
      .map(expected_ctr)
      .astype(float)
)

# Make sure CTR is numeric
df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce")

df["ctr_gap"] = (
    df["expected_ctr"] - df["ctr"]
).clip(lower=0)

# Convert both signals to percentile scores
df["volume_score"] = (
    df["search_volume"]
      .fillna(0)
      .rank(pct=True)
)

df["ctr_gap_score"] = (
    df["ctr_gap"]
      .fillna(0)
      .rank(pct=True)
)

# Baseline score
df["action_score"] = (
    0.5 * df["volume_score"] +
    0.5 * df["ctr_gap_score"]
)

# One reason code and one action
df["reason_code"] = "CTR_OPPORTUNITY"
df["action"] = "REVIEW_CTR"

# Rank highest opportunity first
queue = (
    df[
        [
            "content_id","search_volume","ctr", "avg_position", "expected_ctr", "ctr_gap","action_score","reason_code","action"
        ]
    ]
    .sort_values(
        ["action_score", "search_volume"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue.insert(0, "rank", range(1, len(queue) + 1))

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Saved: {output_path}")
print(f"Rows written: {len(queue):,}")
display(queue.head(10))

Saved: work/outputs/baseline_action_score.csv
Rows written: 30,000


,rank,content_id,search_volume,ctr,avg_position,expected_ctr,ctr_gap,action_score,reason_code,action
0,1,content_201a4a56f4d6,22200.0,0.0,1.0,1.472869,1.472869,0.984708,CTR_OPPORTUNITY,REVIEW_CTR
1,2,content_004d8a5ce838,18100.0,0.0,0.0,1.472869,1.472869,0.984575,CTR_OPPORTUNITY,REVIEW_CTR
2,3,content_137b16f29ca9,5400.0,0.0,0.0,1.472869,1.472869,0.982825,CTR_OPPORTUNITY,REVIEW_CTR
3,4,content_1505ce3bb377,4400.0,0.0,3.0,1.472869,1.472869,0.982317,CTR_OPPORTUNITY,REVIEW_CTR
4,5,content_a31e10779c01,3600.0,0.0,2.0,1.472869,1.472869,0.981633,CTR_OPPORTUNITY,REVIEW_CTR
5,6,content_7ee09c6f28f6,3600.0,0.0,0.0,1.472869,1.472869,0.981633,CTR_OPPORTUNITY,REVIEW_CTR
6,7,content_9107b3dc2bef,3600.0,0.0,2.3,1.472869,1.472869,0.981633,CTR_OPPORTUNITY,REVIEW_CTR
7,8,content_5a894eb9afcd,2900.0,0.0,2.0,1.472869,1.472869,0.980717,CTR_OPPORTUNITY,REVIEW_CTR
8,9,content_ca2662a68168,2900.0,0.0,2.2,1.472869,1.472869,0.980717,CTR_OPPORTUNITY,REVIEW_CTR
9,10,content_ca3223766311,2900.0,0.0,2.6,1.472869,1.472869,0.980717,CTR_OPPORTUNITY,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 ranked pages are reviewed using the assigned action, reason code, confidence note, and a note on what could make each pick incorrect. These are decision-support recommendations and should be reviewed skeptically rather than treated as guaranteed opportunities.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
top20 = queue.head(20).copy()

top20["confidence_note"] = np.select(
    [
        (top20["ctr_gap"] > 0) & (top20["search_volume"] > 0),
        (top20["ctr_gap"] > 0),
        (top20["search_volume"] > 0)
    ],
    [
        "Both CTR gap and volume support the pick.",
        "CTR gap supports the pick, but volume is limited.",
        "Volume supports the pick, but CTR gap is limited."
    ],
    default="Weak signal support."
)

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["search_volume"] <= 0,
        top20["ctr_gap"] <= 0
    ],
    [
        "There may be little search demand despite the ranking.",
        "CTR is not below the typical level for its position."
    ],
    default="The position-bucket average may not represent this page's true search intent."
)

display(
    top20[
        [
            "rank","content_id","action","reason_code","action_score","search_volume",
            "ctr","avg_position","expected_ctr","confidence_note","what_would_make_it_wrong"
        ]
    ]
)

,rank,content_id,action,reason_code,action_score,search_volume,ctr,avg_position,expected_ctr,confidence_note,what_would_make_it_wrong
0,1,content_201a4a56f4d6,REVIEW_CTR,CTR_OPPORTUNITY,0.984708,22200.0,0.0,1.0,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
1,2,content_004d8a5ce838,REVIEW_CTR,CTR_OPPORTUNITY,0.984575,18100.0,0.0,0.0,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
2,3,content_137b16f29ca9,REVIEW_CTR,CTR_OPPORTUNITY,0.982825,5400.0,0.0,0.0,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
3,4,content_1505ce3bb377,REVIEW_CTR,CTR_OPPORTUNITY,0.982317,4400.0,0.0,3.0,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
4,5,content_a31e10779c01,REVIEW_CTR,CTR_OPPORTUNITY,0.981633,3600.0,0.0,2.0,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
5,6,content_7ee09c6f28f6,REVIEW_CTR,CTR_OPPORTUNITY,0.981633,3600.0,0.0,0.0,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
6,7,content_9107b3dc2bef,REVIEW_CTR,CTR_OPPORTUNITY,0.981633,3600.0,0.0,2.3,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
7,8,content_5a894eb9afcd,REVIEW_CTR,CTR_OPPORTUNITY,0.980717,2900.0,0.0,2.0,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
8,9,content_ca2662a68168,REVIEW_CTR,CTR_OPPORTUNITY,0.980717,2900.0,0.0,2.2,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...
9,10,content_ca3223766311,REVIEW_CTR,CTR_OPPORTUNITY,0.980717,2900.0,0.0,2.6,1.472869,Both CTR gap and volume support the pick.,The position-bucket average may not represent ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

#### Weak picks

Some lower-ranked picks may be weak because they have little search volume or no positive CTR gap. These picks should be treated cautiously because the score may not represent a strong opportunity.

#### Leakage check

The baseline score uses only current `search_volume`, `ctr`, and `avg_position`. No product flags, future windows, trend fields, or label-derived inputs were used.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
print("WEAK PICKS")
print("=" * 60)

weak_picks = queue[
    (queue["search_volume"].fillna(0) <= 0) |
    (queue["ctr_gap"].fillna(0) <= 0)
].head(10)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "search_volume",
            "ctr",
            "avg_position",
            "expected_ctr",
            "ctr_gap",
            "action_score",
            "reason_code",
            "action"
        ]
    ]
)

print("\nLEAKAGE CHECK")
print("=" * 60)

rule_features = [
    "search_volume",
    "ctr",
    "avg_position"
]

forbidden_features = [
    "trend_pct",
    "trend_direction"
]

used_forbidden = [
    col for col in forbidden_features
    if col in rule_features
]

print("Features used by rule:", rule_features)
print("Future/trend fields used:", used_forbidden)

if len(used_forbidden) == 0:
    print("Leakage check: PASS")
else:
    print("Leakage check: FAIL")

print("No product flags are used in the score.")
print("No future-window or label-derived inputs are used.")

WEAK PICKS


,rank,content_id,search_volume,ctr,avg_position,expected_ctr,ctr_gap,action_score,reason_code,action
8713,8714,content_1938955b34c4,0.0,0.0,2.9,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8714,8715,content_6cff33402941,0.0,0.0,0.0,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8715,8716,content_fe49ee6b87c4,0.0,0.0,2.6,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8716,8717,content_ea851c8c0ad2,0.0,0.0,2.4,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8717,8718,content_1583937d3411,0.0,0.0,2.5,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8718,8719,content_97804a37420a,0.0,0.0,2.5,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8719,8720,content_a808e255ed8d,0.0,0.0,2.3,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8720,8721,content_ee1b0d51721b,0.0,0.0,2.4,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8721,8722,content_5d77d3077984,0.0,0.0,3.0,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR
8722,8723,content_365cced53e3e,0.0,0.0,2.2,1.472869,1.472869,0.598042,CTR_OPPORTUNITY,REVIEW_CTR



LEAKAGE CHECK
Features used by rule: ['search_volume', 'ctr', 'avg_position']
Future/trend fields used: []
Leakage check: PASS
No product flags are used in the score.
No future-window or label-derived inputs are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.